In [2]:

class Var:
    def __init__(self, name):
        self.name = name

    def __repr__(self):
        return self.name

    def __hash__(self):
        return hash(self.name)

    def __eq__(self, other):
        return isinstance(other, Var) and self.name == other.name


class Func:
    def __init__(self, name, args):
        self.name = name
        self.args = args

    def __repr__(self):
        return f"{self.name}({', '.join(map(str, self.args))})"



def is_variable(x):
    return isinstance(x, Var)



def apply_substitution(term, subst):
    if is_variable(term):
        if term in subst:
            return apply_substitution(subst[term], subst)
        return term

    if isinstance(term, Func):
        return Func(
            term.name,
            [apply_substitution(arg, subst) for arg in term.args]
        )

    return term


def occurs_check(var, term, subst):
    term = apply_substitution(term, subst)

    if var == term:
        return True

    if isinstance(term, Func):
        return any(
            occurs_check(var, arg, subst)
            for arg in term.args
        )

    return False



def unify_terms(x, y, subst):

    x = apply_substitution(x, subst)
    y = apply_substitution(y, subst)


    if x == y:
        return subst


    if is_variable(x):
        if occurs_check(x, y, subst):
            return None

        subst[x] = y
        return subst


    if is_variable(y):
        if occurs_check(y, x, subst):
            return None

        subst[y] = x
        return subst


    if isinstance(x, Func) and isinstance(y, Func):

        if x.name != y.name:
            return None

        if len(x.args) != len(y.args):
            return None

        for a, b in zip(x.args, y.args):
            subst = unify_terms(a, b, subst)

            if subst is None:
                return None

        return subst

    return None



def unify(p1, p2):


    if p1.name != p2.name:
        return None


    if len(p1.args) != len(p2.args):
        return None


    subst = {}


    for a, b in zip(p1.args, p2.args):

        subst = unify_terms(a, b, subst)

        if subst is None:
            return None

    return subst





x = Var("x")
y = Var("y")


p1 = Func("Knows", [x, "John"])


p2 = Func("Knows", ["Mary", y])

print("Expression 1:", p1)
print("Expression 2:", p2)


result = unify(p1, p2)

if result is None:

    print("\nUnification: FAILURE")
    print("SUBST = NIL")

else:

    print("\nUnification: SUCCESS")


    print("\nSubstitution Set (SUBST):")

    if not result:
        print("NIL")
    else:
        for var, value in result.items():
            print("{" + str(var) + "/" + str(value) + "}")

    expr1 = apply_substitution(p1, result)
    expr2 = apply_substitution(p2, result)


    print("\nFinal Unified Expressions:")
    print("Expression 1:", expr1)
    print("Expression 2:", expr2)


    if expr1 == expr2:
        print("\nBoth expressions are unified successfully.")


Expression 1: Knows(x, John)
Expression 2: Knows(Mary, y)

Unification: SUCCESS

Substitution Set (SUBST):
{x/Mary}
{y/John}

Final Unified Expressions:
Expression 1: Knows(Mary, John)
Expression 2: Knows(Mary, John)
